# Train ST-GCN on NSLT Landmark Tensors

This notebook trains a compact ST-GCN model from the preprocessed landmark dataset.

Expected dataset structure:

```text
preprocessed_nslt300/
  landmarks/
  train.csv
  val.csv
  test.csv
  label_map.csv
  stgcn_graph.json
```

Each sample is expected to have shape `(3, 32, 120, 1)`.

In [1]:
from pathlib import Path
import csv
import json
import random
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

torch: 2.10.0+cu126
cuda available: True
gpu: NVIDIA GeForce RTX 2050


## Config

On Kaggle, add your preprocessed folder as a Dataset, then update `DATA_ROOT` if needed.

Common examples:

```python
DATA_ROOT = Path('/kaggle/input/YOUR-DATASET/preprocessed_nslt300')
DATA_ROOT = Path('/kaggle/working/preprocessed_nslt300')
```

In [ ]:
DATA_ROOT = Path('./preprocessed_nslt300')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('preprocessed_nslt300')

TRAIN_CSV = DATA_ROOT / 'train.csv'
VAL_CSV = DATA_ROOT / 'val.csv'
TEST_CSV = DATA_ROOT / 'test.csv'
LABEL_MAP = DATA_ROOT / 'label_map.csv'
GRAPH_JSON = DATA_ROOT / 'stgcn_graph.json'

CHECKPOINT_DIR = Path('./checkpoints/nslt300_stgcn')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 5e-4
DROPOUT = 0.3
NUM_WORKERS = 2
USE_MASK = True
USE_AMP = True
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DATA_ROOT:', DATA_ROOT)
print('DEVICE:', DEVICE)
print('train exists:', TRAIN_CSV.exists())
print('val exists:', VAL_CSV.exists())
print('graph exists:', GRAPH_JSON.exists())

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True

## Dataset Sanity Check

In [ ]:
def read_rows(csv_path):
    with open(csv_path, 'r', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def resolve_data_path(path_text):
    path = Path(path_text)
    if path.exists():
        return path

    # CSV paths were created locally as preprocessed_nslt300/landmarks/xxx.npy.
    # On Kaggle, DATA_ROOT points directly to that preprocessed_nslt300 folder.
    parts = list(path.parts)
    if DATA_ROOT.name in parts:
        relative_inside_data_root = Path(*parts[parts.index(DATA_ROOT.name) + 1:])
        candidate = DATA_ROOT / relative_inside_data_root
        if candidate.exists():
            return candidate

    candidate = DATA_ROOT / path
    if candidate.exists():
        return candidate

    candidate = DATA_ROOT / 'landmarks' / path.name
    return candidate

train_rows = read_rows(TRAIN_CSV)
val_rows = read_rows(VAL_CSV)
test_rows = read_rows(TEST_CSV) if TEST_CSV.exists() else []

print('train samples:', len(train_rows))
print('val samples:', len(val_rows))
print('test samples:', len(test_rows))
print('train labels:', len(set(row['label'] for row in train_rows)))
print('val labels:', len(set(row['label'] for row in val_rows)))

sample = train_rows[0]
x = np.load(resolve_data_path(sample['feature_path']))
m = np.load(resolve_data_path(sample['mask_path']))
print('sample video:', sample['video_id'], sample['gloss'], sample['label'])
print('x shape:', x.shape, x.dtype)
print('mask shape:', m.shape, m.dtype)
print('visible ratio:', float(m.mean()))

If the previous cell fails on Kaggle because paths inside the CSV are relative, run this path-fix cell.

In [ ]:
def resolve_data_path(path_text):
    path = Path(path_text)
    if path.exists():
        return path
    candidate = DATA_ROOT.parent / path
    if candidate.exists():
        return candidate
    candidate = DATA_ROOT / path.name
    if candidate.exists():
        return candidate
    candidate = DATA_ROOT / 'landmarks' / path.name
    return candidate

# This function is used in the Dataset below, so you usually do not need to edit CSV files.

## Dataset and Graph

In [ ]:
class LandmarkDataset(Dataset):
    def __init__(self, csv_path, use_mask=True):
        self.csv_path = Path(csv_path)
        self.use_mask = use_mask
        self.rows = read_rows(self.csv_path)
        if not self.rows:
            raise ValueError(f'No rows found in {self.csv_path}')

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        feature_path = resolve_data_path(row['feature_path'])
        x = np.load(feature_path).astype(np.float32)

        if self.use_mask:
            mask_path = resolve_data_path(row['mask_path'])
            mask = np.load(mask_path).astype(np.float32)
            x = x * mask[np.newaxis, :, :, np.newaxis]

        y = int(row['label'])
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)


def load_graph(graph_path):
    with open(graph_path, 'r', encoding='utf-8') as f:
        graph = json.load(f)

    num_vertices = int(graph['num_vertices'])
    adjacency = np.eye(num_vertices, dtype=np.float32)
    for a, b in graph['edges']:
        adjacency[a, b] = 1.0
        adjacency[b, a] = 1.0

    degree = adjacency.sum(axis=1)
    degree_inv_sqrt = np.power(degree, -0.5, where=degree > 0)
    degree_inv_sqrt[degree == 0] = 0.0
    normalized = degree_inv_sqrt[:, None] * adjacency * degree_inv_sqrt[None, :]
    return torch.from_numpy(normalized), graph


def infer_num_classes(label_map_path, train_csv_path):
    labels = []
    if Path(label_map_path).exists():
        with open(label_map_path, 'r', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                labels.append(int(row['label']))
    if not labels:
        with open(train_csv_path, 'r', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                labels.append(int(row['label']))
    return max(labels) + 1


adjacency, graph_info = load_graph(GRAPH_JSON)
NUM_CLASSES = infer_num_classes(LABEL_MAP, TRAIN_CSV)
print('adjacency:', adjacency.shape)
print('classes:', NUM_CLASSES)
print('graph shape:', graph_info['shape'])

## Model

In [ ]:
class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency, stride=1, dropout=0.0):
        super().__init__()
        self.register_buffer('adjacency', adjacency)

        self.gcn = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=(9, 1),
                stride=(stride, 1),
                padding=(4, 0),
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout),
        )

        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.residual(x)
        x = torch.einsum('nctv,vw->nctw', x, self.adjacency)
        x = self.gcn(x)
        x = self.tcn(x)
        return self.relu(x + residual)


class CompactSTGCN(nn.Module):
    def __init__(self, num_classes, adjacency, in_channels=3, dropout=0.3):
        super().__init__()
        self.data_bn = nn.BatchNorm1d(in_channels * adjacency.shape[0])
        self.blocks = nn.Sequential(
            STGCNBlock(in_channels, 64, adjacency, dropout=dropout),
            STGCNBlock(64, 64, adjacency, dropout=dropout),
            STGCNBlock(64, 128, adjacency, stride=2, dropout=dropout),
            STGCNBlock(128, 128, adjacency, dropout=dropout),
            STGCNBlock(128, 256, adjacency, stride=2, dropout=dropout),
            STGCNBlock(256, 256, adjacency, dropout=dropout),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        # Input: N, C, T, V, M
        n, c, t, v, m = x.shape
        x = x.permute(0, 4, 3, 1, 2).contiguous()
        x = x.view(n * m, v * c, t)
        x = self.data_bn(x)
        x = x.view(n, m, v, c, t)
        x = x.permute(0, 1, 3, 4, 2).contiguous()
        x = x.view(n * m, c, t, v)

        x = self.blocks(x)
        x = x.mean(dim=(2, 3))
        x = x.view(n, m, -1).mean(dim=1)
        return self.classifier(x)

## Training Helpers

In [ ]:
def accuracy_topk(logits, target, topk=(1, 5)):
    max_k = max(topk)
    _, pred = logits.topk(max_k, dim=1)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    results = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum()
        results.append(correct_k.mul_(100.0 / target.size(0)).item())
    return results


def run_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total_samples = 0

    for x, y in tqdm(loader, desc='train', leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            logits = model(x)
            loss = criterion(logits, y)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size = y.size(0)
        top1, top5 = accuracy_topk(logits.detach(), y)
        total_loss += loss.item() * batch_size
        total_top1 += top1 * batch_size
        total_top5 += top5 * batch_size
        total_samples += batch_size

    return {
        'loss': total_loss / total_samples,
        'top1': total_top1 / total_samples,
        'top5': total_top5 / total_samples,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device, split_name='val'):
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total_samples = 0

    for x, y in tqdm(loader, desc=split_name, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        batch_size = y.size(0)
        top1, top5 = accuracy_topk(logits, y)
        total_loss += loss.item() * batch_size
        total_top1 += top1 * batch_size
        total_top5 += top5 * batch_size
        total_samples += batch_size

    return {
        'loss': total_loss / total_samples,
        'top1': total_top1 / total_samples,
        'top5': total_top5 / total_samples,
    }


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_top1, config):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict() if scheduler else None,
            'best_top1': best_top1,
            'config': config,
        },
        path,
    )

## Build Loaders and Model

In [ ]:
train_dataset = LandmarkDataset(TRAIN_CSV, use_mask=USE_MASK)
val_dataset = LandmarkDataset(VAL_CSV, use_mask=USE_MASK)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == 'cuda',
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == 'cuda',
)

adjacency = adjacency.to(DEVICE)
model = CompactSTGCN(NUM_CLASSES, adjacency, dropout=DROPOUT).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler() if USE_AMP and DEVICE.type == 'cuda' else None

num_params = sum(p.numel() for p in model.parameters())
print('parameters:', f'{num_params:,}')
print('amp:', scaler is not None)

xb, yb = next(iter(train_loader))
print('batch x:', xb.shape)
print('batch y:', yb.shape)
with torch.no_grad():
    out = model(xb.to(DEVICE))
print('logits:', out.shape)

## Train

In [ ]:
best_top1 = 0.0
history = []
config = {
    'data_root': str(DATA_ROOT),
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'dropout': DROPOUT,
    'use_mask': USE_MASK,
    'use_amp': USE_AMP,
    'seed': SEED,
}

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler=scaler)
    val_metrics = evaluate(model, val_loader, criterion, DEVICE, split_name='val')
    scheduler.step()

    row = {
        'epoch': epoch,
        **{f'train_{k}': v for k, v in train_metrics.items()},
        **{f'val_{k}': v for k, v in val_metrics.items()},
    }
    history.append(row)

    print(
        f"epoch {epoch:03d}/{EPOCHS} "
        f"train_loss={train_metrics['loss']:.4f} "
        f"train_top1={train_metrics['top1']:.2f} "
        f"train_top5={train_metrics['top5']:.2f} "
        f"val_loss={val_metrics['loss']:.4f} "
        f"val_top1={val_metrics['top1']:.2f} "
        f"val_top5={val_metrics['top5']:.2f}"
    )

    save_checkpoint(CHECKPOINT_DIR / 'last.pt', model, optimizer, scheduler, epoch, best_top1, config)

    if val_metrics['top1'] > best_top1:
        best_top1 = val_metrics['top1']
        save_checkpoint(CHECKPOINT_DIR / 'best.pt', model, optimizer, scheduler, epoch, best_top1, config)
        print(f'saved new best: top1={best_top1:.2f}')

## Save Training History

In [ ]:

history_path = Path('/nslt300_stgcn_history.csv')
if history:
    with open(history_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)
    print('saved history:', history_path)
print('best checkpoint:', CHECKPOINT_DIR / 'best.pt')
print('last checkpoint:', CHECKPOINT_DIR / 'last.pt')

## Optional Test Evaluation

Run this only after you are happy with validation performance.

In [ ]:
if TEST_CSV.exists() and (CHECKPOINT_DIR / 'best.pt').exists():
    checkpoint = torch.load(CHECKPOINT_DIR / 'best.pt', map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    test_dataset = LandmarkDataset(TEST_CSV, use_mask=USE_MASK)
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda',
    )
    test_metrics = evaluate(model, test_loader, criterion, DEVICE, split_name='test')
    print(test_metrics)


#top 1 9% and top 5 26%